# Batch Agent Runner — Async Parallel Task Processing
## Many Tasks at Once — Async Concurrency for Agent Workflows
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/72-batch-agent-runner/batch_agent_runner_workbook.ipynb)

Runs 9 independent LLM tasks in batches of 3 using asyncio.gather and ainvoke. Includes exponential-backoff retry on failure. Contrasts sequential vs concurrent wall-clock time.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Sequential vs concurrent LLM calls; asyncio.gather for parallelism |
| 2 | **process_task()** — ainvoke with MAX_RETRIES=3 and exponential backoff |
| 3 | **Batching** — Split tasks into BATCH_SIZE=3 groups; gather within each batch |
| 4 | **Timing Comparison** — Sequential baseline vs async batch — measure wall-clock time |
| 5 | **Full Runner** — run_batch_agent() over all 9 tasks |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
import asyncio, time
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

BATCH_SIZE = 3
MAX_RETRIES = 3
RETRY_DELAY = 1.0

SAMPLE_TASKS = [
    "What is photosynthesis? One sentence.",
    "What is the capital of Japan? One sentence.",
    "What is Newton's first law? One sentence.",
    "What is DNA? One sentence.",
    "What is the speed of light? One sentence.",
    "What is machine learning? One sentence.",
    "What is blockchain? One sentence.",
    "What is quantum computing? One sentence.",
    "What is the Pythagorean theorem? One sentence.",
]

async def process_task(llm, task: str, index: int) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            r = await llm.ainvoke([HumanMessage(content=task)])
            return {"index": index, "task": task[:40], "answer": r.content.strip(), "status": "ok", "attempts": attempt+1}
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                await asyncio.sleep(RETRY_DELAY * (2**attempt))
            else:
                return {"index": index, "task": task[:40], "answer": "", "status": f"error: {e}", "attempts": attempt+1}

async def run_batch_agent(tasks):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    results = []
    batches = [tasks[i:i+BATCH_SIZE] for i in range(0, len(tasks), BATCH_SIZE)]
    for bi, batch in enumerate(batches):
        print(f"  Batch {bi+1}/{len(batches)} ({len(batch)} tasks)...")
        batch_results = await asyncio.gather(*[process_task(llm, t, i+bi*BATCH_SIZE) for i, t in enumerate(batch)])
        results.extend(batch_results)
    return results

start = time.time()
results = asyncio.run(run_batch_agent(SAMPLE_TASKS))
elapsed = time.time() - start

print(f"\nCompleted {len(results)} tasks in {elapsed:.1f}s")
print(f"Avg {elapsed/len(results):.1f}s/task (vs ~{elapsed:.0f}s sequential)")
for r in results:
    print(f"  [{r['status']}] {r['task']}: {r['answer'][:80]}")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*